In [ ]:
#!/usr/bin/env python3
"""
Scheduler vs Cloud Analysis — Rubin USDF version
=================================================
Combines:
  - Cloud attenuation maps from feb5_data.csv (DREAM HDF5 files)
  - Scheduler pointings from baseline_v5.1.0_10yrs.db (SQLite, all nights)
  - Binning at 4x4 degrees (>= Rubin FoV, slightly over 3.5 deg)
  - Patchy-cloud prioritization: SD > 0.15 mag AND mean > 0.1 mag
  - Full combinatorics: every scheduler night x every DREAM cloud night
  - Parallel processing across (scheduler_night, cloud_date) pairs
  - Incremental CSV saves after each completed night

Outputs:
    results/gain_<sched_night>_<cloud_date>.csv   -- per-pair results
    results/summary.csv                           -- rolling summary table
    scheduler_gain_analysis.png
    scheduler_gain_timeseries.png
"""

import io
import os
import sqlite3
import warnings
import traceback
from datetime import datetime, timezone
from pathlib import Path
from multiprocessing import Pool, cpu_count, current_process

import numpy as np
import pandas as pd
import h5py
import healpy as hp
import matplotlib
matplotlib.use("Agg")  # headless on USDF
import matplotlib.pyplot as plt
import seaborn as sns
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u
from lsst.resources import ResourcePath

# ── Configuration ─────────────────────────────────────────────────────────────
CLOUD_CSV       = "feb5_data.csv"
DB_FILE         = "baseline_v5.1.0_10yrs.db"
RESULTS_DIR     = Path("results")
SUMMARY_CSV     = RESULTS_DIR / "summary.csv"

NSIDE_EXPECTED  = 32
NEST            = True
RUBIN_LAT       = -30.244639
RUBIN_LON       = -70.749417
RUBIN_HEIGHT_M  = 2663.0

# Rubin FoV binning: 4 deg x 4 deg bins (>= 3.5 deg FoV diameter)
RUBIN_BIN_SIZE  = 4   # degrees per bin in the 1-deg-res alt-az grid
N_AZ_FINE       = 360
N_ALT_FINE      = 90

# Patchy cloud thresholds
PATCHY_SD_THRESH   = 0.15  # mag
PATCHY_MEAN_THRESH = 0.10  # mag

# Parallelism — USDF: leave some headroom
N_WORKERS = min(32, max(1, cpu_count() - 4))

# Time-matching window between scheduler pointing and cloud map (minutes)
TIME_WINDOW_MIN = 15.0

# ── Utilities ─────────────────────────────────────────────────────────────────

def transform_url(url: str) -> str:
    url = str(url).strip()
    if url.startswith("https://s3.cp.lsst.org/"):
        return url.replace("https://s3.cp.lsst.org/", "s3://lfa@")
    return url


def fetch_map(url: str) -> np.ndarray:
    """Fetch a DREAM HDF5 cloud map and return the 'clouds' array."""
    rp = ResourcePath(url)
    with rp.open("rb") as fd:
        data = fd.read()
    bio = io.BytesIO(data)
    with h5py.File(bio, "r") as f:
        if "clouds" not in f:
            raise RuntimeError(f"'clouds' dataset missing in {url}")
        arr = np.array(f["clouds"]).ravel()
    return arr


def healpix_to_altaz(mp: np.ndarray, nside: int, obstime) -> tuple:
    """
    Project a HEALPix map (NESTED) from ICRS to Alt/Az at obstime.
    Returns (az_deg, alt_deg, values).
    """
    npix = hp.nside2npix(nside)
    pix  = np.arange(npix)
    theta, phi = hp.pix2ang(nside, pix, nest=NEST)
    ra  = np.degrees(phi)
    dec = 90.0 - np.degrees(theta)

    if not isinstance(obstime, Time):
        obstime = Time(obstime)
    obstime = obstime.utc

    location = EarthLocation(lat=RUBIN_LAT * u.deg,
                             lon=RUBIN_LON  * u.deg,
                             height=RUBIN_HEIGHT_M * u.m)
    sky   = SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs")
    altaz = sky.transform_to(AltAz(obstime=obstime, location=location))

    vals = np.asarray(mp, dtype=float)
    vals = np.where(vals == hp.UNSEEN, np.nan, vals)
    return altaz.az.deg % 360.0, altaz.alt.deg, vals


def bin_altaz_to_rubin_fov(az_deg: np.ndarray, alt_deg: np.ndarray,
                            vals: np.ndarray,
                            bin_size: int = RUBIN_BIN_SIZE) -> tuple:
    """
    Bin 1-deg-resolution alt-az data into bin_size x bin_size degree cells.
    This matches approximately the Rubin FoV (3.5 deg diameter -> 4 deg bin).
    Returns (az_centers, alt_centers, mean_values).
    """
    grid   = np.full((N_ALT_FINE, N_AZ_FINE), np.nan)
    counts = np.zeros((N_ALT_FINE, N_AZ_FINE), dtype=np.int32)

    for i in range(len(az_deg)):
        if np.isnan(vals[i]):
            continue
        az_idx  = int(az_deg[i]  % N_AZ_FINE)
        alt_idx = int(alt_deg[i])
        if 0 <= alt_idx < N_ALT_FINE and 0 <= az_idx < N_AZ_FINE:
            if np.isnan(grid[alt_idx, az_idx]):
                grid[alt_idx, az_idx] = vals[i]
            else:
                grid[alt_idx, az_idx] += vals[i]
            counts[alt_idx, az_idx] += 1

    # Normalize accumulated sums
    mask = counts > 0
    grid[mask] /= counts[mask]

    n_alt_b = N_ALT_FINE // bin_size
    n_az_b  = N_AZ_FINE  // bin_size

    az_out, alt_out, val_out = [], [], []
    for ia in range(n_alt_b):
        for ja in range(n_az_b):
            block = grid[ia*bin_size:(ia+1)*bin_size,
                         ja*bin_size:(ja+1)*bin_size]
            if not np.all(np.isnan(block)):
                val_out.append(np.nanmean(block))
                az_out.append((ja + 0.5) * bin_size)
                alt_out.append((ia + 0.5) * bin_size)

    return np.array(az_out), np.array(alt_out), np.array(val_out)


def classify_patchy(vals_above_horizon: np.ndarray) -> bool:
    """
    Return True if this cloud map is patchy:
      - spatial SD > PATCHY_SD_THRESH
      - mean attenuation > PATCHY_MEAN_THRESH
    """
    clean = vals_above_horizon[~np.isnan(vals_above_horizon)]
    if len(clean) < 5:
        return False
    return (np.std(clean)  > PATCHY_SD_THRESH and
            np.mean(clean) > PATCHY_MEAN_THRESH)


def radec_to_altaz(ra, dec, obstime):
    if not isinstance(obstime, Time):
        obstime = Time(obstime)
    obstime = obstime.utc
    location = EarthLocation(lat=RUBIN_LAT * u.deg,
                             lon=RUBIN_LON  * u.deg,
                             height=RUBIN_HEIGHT_M * u.m)
    sky   = SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs")
    altaz = sky.transform_to(AltAz(obstime=obstime, location=location))
    return altaz.alt.deg, altaz.az.deg % 360.0


# ── Data Loaders ──────────────────────────────────────────────────────────────

def load_cloud_index(csv_file: str) -> pd.DataFrame:
    """
    Load the cloud CSV index.
    Expects columns: time, <url_column>.
    Returns DataFrame with columns: time (UTC-aware), url (transformed), date.
    """
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.replace('"', '').str.strip()
    url_col = [c for c in df.columns if c.lower() != "time"][0]
    df = df.dropna(subset=[url_col]).copy()
    df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True)
    df = df.dropna(subset=["time"])
    df[url_col] = df[url_col].astype(str).str.strip().apply(transform_url)
    df = df.rename(columns={url_col: "url"})
    df["date"] = df["time"].dt.date
    return df.sort_values("time").reset_index(drop=True)


def load_all_scheduler_nights(db_file: str) -> dict:
    """
    Load all nights from the SQLite scheduler DB.
    Returns dict: {night_int: DataFrame}.
    Pulls only the columns needed to keep memory reasonable.
    """
    conn = sqlite3.connect(db_file)
    obs_df = pd.read_sql_query(
        "SELECT observationStartMJD, fieldRA, fieldDec, band, night "
        "FROM observations ORDER BY observationStartMJD",
        conn)
    conn.close()
    obs_df["night"] = obs_df["night"].astype(int)
    return {n: grp.reset_index(drop=True)
            for n, grp in obs_df.groupby("night")}


# ── Core worker: one (scheduler_night, cloud_date) pair ──────────────────────

def analyze_pair(args):
    """
    Worker: analyze one (sched_night_id, cloud_date) pair.
    Returns list of result dicts.
    """
    (sched_night_id, sched_df, cloud_date, cloud_rows,
     results_dir_str) = args

    pid     = current_process().name
    out_csv = Path(results_dir_str) / f"gain_{sched_night_id}_{cloud_date}.csv"

    # Resume support: skip if already computed
    if out_csv.exists():
        try:
            existing = pd.read_csv(out_csv)
            print(f"[{pid}] SKIP (cached): night={sched_night_id} "
                  f"date={cloud_date}  ({len(existing)} rows)")
            return existing.to_dict("records")
        except Exception:
            pass  # Re-compute if file is corrupt

    results = []

    # Pre-compute scheduler fractional UTC hours for vectorised matching
    try:
        sched_times_astropy = Time(sched_df["observationStartMJD"].values, format="mjd")
        sched_dt = sched_times_astropy.to_datetime(timezone=timezone.utc)
        sched_frac_h = np.array([
            t.hour + t.minute / 60.0 + t.second / 3600.0
            for t in sched_dt])
    except Exception as e:
        print(f"[{pid}] ✗ night={sched_night_id}: scheduler time conversion failed: {e}")
        return []

    for _, cloud_row in cloud_rows.iterrows():
        cloud_time = cloud_row["time"]
        url        = cloud_row["url"]

        # Fetch map
        try:
            mp = fetch_map(url)
        except Exception as e:
            print(f"[{pid}]   ✗ fetch failed {url}: {e}")
            continue

        # Project to alt/az
        try:
            az, alt, vals = healpix_to_altaz(mp, NSIDE_EXPECTED, cloud_time)
        except Exception as e:
            print(f"[{pid}]   ✗ projection failed: {e}")
            continue

        above = alt > 0
        if not np.any(above & ~np.isnan(vals)):
            continue

        # Patchy classification (pre-binning, full resolution)
        is_patchy = classify_patchy(vals[above])

        # Bin to Rubin FoV resolution
        az_b, alt_b, vals_b = bin_altaz_to_rubin_fov(az[above], alt[above], vals[above])
        if len(vals_b) == 0:
            continue

        map_mean = float(np.nanmean(vals_b))
        map_sd   = float(np.nanstd(vals_b))
        best_val = float(np.nanmin(vals_b))
        best_idx = int(np.nanargmin(vals_b))
        best_az  = float(az_b[best_idx])
        best_alt = float(alt_b[best_idx])

        # Time-of-day fractional hour for this cloud map
        ct = cloud_time
        cloud_frac_h = ct.hour + ct.minute / 60.0 + ct.second / 3600.0

        # Match to nearest scheduler observation within window
        diff_h = np.abs(sched_frac_h - cloud_frac_h)
        diff_h = np.minimum(diff_h, 24.0 - diff_h)  # wrap midnight

        window_h  = TIME_WINDOW_MIN / 60.0
        cand_mask = diff_h < window_h
        if not np.any(cand_mask):
            continue

        best_cand_idx = int(np.argmin(np.where(cand_mask, diff_h, np.inf)))
        obs           = sched_df.iloc[best_cand_idx]
        time_off_min  = float(diff_h[best_cand_idx] * 60.0)

        # Convert RA/Dec of actual pointing to alt/az
        try:
            actual_alt, actual_az = radec_to_altaz(
                obs["fieldRA"], obs["fieldDec"], cloud_time)
        except Exception:
            continue

        if actual_alt <= 0:
            continue

        # Nearest binned cell to actual pointing
        dists      = np.sqrt((az_b - actual_az)**2 + (alt_b - actual_alt)**2)
        nearest    = int(np.nanargmin(dists))
        actual_val = float(vals_b[nearest])

        if np.isnan(actual_val):
            continue

        gain = actual_val - best_val  # positive = scheduler worse than best

        results.append({
            "sched_night":      sched_night_id,
            "cloud_date":       str(cloud_date),
            "cloud_time_utc":   cloud_time.isoformat(),
            "is_patchy":        bool(is_patchy),
            "map_mean_mag":     round(map_mean,   4),
            "map_sd_mag":       round(map_sd,     4),
            "best_val_mag":     round(best_val,   4),
            "best_az_deg":      round(best_az,    2),
            "best_alt_deg":     round(best_alt,   2),
            "actual_val_mag":   round(actual_val, 4),
            "actual_az_deg":    round(actual_az,  2),
            "actual_alt_deg":   round(actual_alt, 2),
            "gain_mag":         round(gain,        4),
            "time_offset_min":  round(time_off_min, 2),
            "band":             obs.get("band", "?"),
        })

    # Save per-pair CSV immediately (resume support + incremental output)
    if results:
        pd.DataFrame(results).to_csv(out_csv, index=False)
        n_patchy   = sum(r["is_patchy"] for r in results)
        mean_gain  = np.mean([r["gain_mag"] for r in results])
        print(f"[{pid}] night={sched_night_id} date={cloud_date}: "
              f"{len(results)} matches, {n_patchy} patchy | "
              f"mean gain={mean_gain:.3f} mag -> {out_csv.name}")
    else:
        print(f"[{pid}] night={sched_night_id} date={cloud_date}: no matches")

    return results


# ── Summary ───────────────────────────────────────────────────────────────────

def update_summary(all_results: list, summary_csv: Path):
    if not all_results:
        return
    df = pd.DataFrame(all_results)
    summary = (df.groupby(["sched_night", "cloud_date", "is_patchy"])
                 .agg(
                     n_obs       =("gain_mag", "count"),
                     mean_gain   =("gain_mag", "mean"),
                     median_gain =("gain_mag", "median"),
                     std_gain    =("gain_mag", "std"),
                     max_gain    =("gain_mag", "max"),
                     min_gain    =("gain_mag", "min"),
                     pct_worse_5 =("gain_mag", lambda x: (x > 0.05).mean() * 100),
                     mean_map_sd =("map_sd_mag", "mean"),
                 )
                 .reset_index()
                 .sort_values(["sched_night", "cloud_date"]))
    summary.to_csv(summary_csv, index=False)
    print(f"  [summary] -> {summary_csv}  ({len(summary)} rows)")


# ── Plots ─────────────────────────────────────────────────────────────────────

def create_plots(all_results: list):
    if not all_results:
        print("No results to plot.")
        return

    df        = pd.DataFrame(all_results)
    patchy_df = df[df["is_patchy"]]

    sns.set_style("whitegrid")

    # ── Plot 1: four-panel summary ────────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(18, 13))

    # 1a. Boxplots by scheduler night: all vs patchy
    ax     = axes[0, 0]
    nights = sorted(df["sched_night"].unique())
    pos_all    = np.arange(len(nights)) * 2.5
    pos_patchy = pos_all + 0.9
    bp1 = ax.boxplot([df[df["sched_night"]==n]["gain_mag"].values for n in nights],
                     positions=pos_all,    widths=0.7, patch_artist=True, showfliers=False)
    bp2 = ax.boxplot([patchy_df[patchy_df["sched_night"]==n]["gain_mag"].values
                      if len(patchy_df[patchy_df["sched_night"]==n]) > 0
                      else [np.nan] for n in nights],
                     positions=pos_patchy, widths=0.7, patch_artist=True, showfliers=False)
    for p in bp1["boxes"]: p.set_facecolor("lightblue")
    for p in bp2["boxes"]: p.set_facecolor("salmon")
    ax.set_xticks(pos_all + 0.45)
    ax.set_xticklabels([str(n) for n in nights], rotation=45, fontsize=8)
    ax.axhline(0, color="red", linestyle="--", lw=1.5)
    ax.set_ylabel("Gain [mag]  (actual - best)", fontsize=11)
    ax.set_title("Gain by Scheduler Night\nBlue=all maps, Red=patchy only", fontsize=12)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor="lightblue", label="All maps"),
                        Patch(facecolor="salmon",    label="Patchy maps")], fontsize=9)

    # 1b. Mean gain by cloud date (patchy only)
    ax    = axes[0, 1]
    dates = sorted(patchy_df["cloud_date"].unique()) if len(patchy_df) > 0 else []
    if dates:
        means = [patchy_df[patchy_df["cloud_date"]==d]["gain_mag"].mean() for d in dates]
        stds  = [patchy_df[patchy_df["cloud_date"]==d]["gain_mag"].std()  for d in dates]
        bars  = ax.bar(range(len(dates)), means, yerr=stds, capsize=4,
                       alpha=0.7, color="salmon")
        ax.set_xticks(range(len(dates)))
        ax.set_xticklabels(dates, rotation=45, fontsize=8)
        for bar, m in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f"{m:.3f}", ha="center", va="bottom", fontsize=7)
    ax.axhline(0, color="red", linestyle="--", lw=1.5)
    ax.set_ylabel("Mean Gain [mag]", fontsize=11)
    ax.set_title("Mean Gain by Cloud Date\n(Patchy maps only)", fontsize=12)

    # 1c. Heatmap: sched_night x cloud_date (patchy only)
    ax = axes[1, 0]
    if len(patchy_df) > 0:
        pivot = (patchy_df.groupby(["sched_night", "cloud_date"])["gain_mag"]
                          .mean().unstack(fill_value=np.nan))
        im = ax.imshow(pivot.values, cmap="RdYlGn_r", aspect="auto",
                       vmin=-0.1, vmax=0.5)
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_yticks(range(len(pivot.index)))
        ax.set_xticklabels(pivot.columns, rotation=45, fontsize=7)
        ax.set_yticklabels(pivot.index, fontsize=8)
        ax.set_xlabel("Cloud Date", fontsize=11)
        ax.set_ylabel("Scheduler Night", fontsize=11)
        ax.set_title("Mean Gain Heatmap [mag] - Patchy Only\n(Red=worse, Green=better)",
                     fontsize=12)
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                v = pivot.values[i, j]
                if not np.isnan(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                            fontsize=7)
        plt.colorbar(im, ax=ax, label="Mean Gain [mag]")

    # 1d. Overall gain histogram: all vs patchy
    ax = axes[1, 1]
    ax.hist(df["gain_mag"].dropna(),        bins=60, alpha=0.5,
            color="steelblue", label=f"All ({len(df)})",     histtype="stepfilled")
    ax.hist(patchy_df["gain_mag"].dropna(), bins=60, alpha=0.6,
            color="salmon",    label=f"Patchy ({len(patchy_df)})", histtype="stepfilled")
    ax.axvline(0, color="red", linestyle="--", lw=1.5)
    ax.set_xlabel("Gain [mag]", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.set_title("Overall Gain Distribution", fontsize=12)
    ax.legend(fontsize=10)

    plt.suptitle(
        f"Scheduler vs Best-Sky Cloud Analysis\n"
        f"FoV bin: {RUBIN_BIN_SIZE}dx{RUBIN_BIN_SIZE}d | "
        f"{len(df)} total, {len(patchy_df)} patchy matches",
        fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("scheduler_gain_analysis.png", dpi=150, bbox_inches="tight")
    print("Saved: scheduler_gain_analysis.png")

    # ── Plot 2: time-series per cloud date (patchy only) ──────────────────────
    if len(patchy_df) > 0:
        cloud_dates = sorted(patchy_df["cloud_date"].unique())
        fig2, axes2 = plt.subplots(len(cloud_dates), 1,
                                   figsize=(14, 4 * len(cloud_dates)), squeeze=False)
        for i, cdate in enumerate(cloud_dates):
            ax2  = axes2[i, 0]
            sub  = patchy_df[patchy_df["cloud_date"] == cdate]
            tseries = pd.to_datetime(sub["cloud_time_utc"])
            hours   = tseries.dt.hour + tseries.dt.minute / 60.0
            sc = ax2.scatter(hours, sub["gain_mag"], c=sub["map_sd_mag"],
                             cmap="plasma", alpha=0.6, s=15,
                             vmin=PATCHY_SD_THRESH, vmax=0.5)
            plt.colorbar(sc, ax=ax2, label="Map SD [mag]")
            ax2.axhline(0, color="red", linestyle="--", lw=1.5)
            ax2.set_ylabel("Gain [mag]", fontsize=10)
            ax2.set_xlabel("UTC Hour", fontsize=10)
            ax2.set_title(f"Cloud Date: {cdate} -- patchy only  |  "
                          f"mean gain={sub['gain_mag'].mean():.3f} mag  "
                          f"n={len(sub)}",
                          fontsize=11, fontweight="bold")
            ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig("scheduler_gain_timeseries.png", dpi=150, bbox_inches="tight")
        print("Saved: scheduler_gain_timeseries.png")


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    RESULTS_DIR.mkdir(exist_ok=True)

    print("=" * 70)
    print("  SCHEDULER vs CLOUD ANALYSIS  (Rubin USDF)")
    print(f"  Workers  : {N_WORKERS}")
    print(f"  FoV bin  : {RUBIN_BIN_SIZE} deg x {RUBIN_BIN_SIZE} deg")
    print(f"  Patchy   : SD > {PATCHY_SD_THRESH} mag  &  mean > {PATCHY_MEAN_THRESH} mag")
    print(f"  Time win : {TIME_WINDOW_MIN} min")
    print("=" * 70)

    # 1. Load cloud index
    print(f"\n[1/4] Loading cloud index from {CLOUD_CSV} ...")
    cloud_df    = load_cloud_index(CLOUD_CSV)
    cloud_dates = sorted(cloud_df["date"].unique())
    print(f"      {len(cloud_df)} maps across {len(cloud_dates)} dates: "
          f"{cloud_dates[0]} to {cloud_dates[-1]}")

    # 2. Load scheduler nights
    print(f"\n[2/4] Loading scheduler nights from {DB_FILE} ...")
    nights_dict = load_all_scheduler_nights(DB_FILE)
    print(f"      {len(nights_dict)} nights (#{min(nights_dict)} to #{max(nights_dict)})")

    # 3. Pre-classify cloud dates as patchy / not patchy
    print(f"\n[3/4] Pre-classifying {len(cloud_dates)} cloud dates for patchiness ...")
    patchy_dates     = []
    non_patchy_dates = []

    for cdate, cgroup in cloud_df.groupby("date"):
        date_is_patchy = False
        for _, crow in cgroup.iterrows():
            try:
                mp  = fetch_map(crow["url"])
                az, alt, vals = healpix_to_altaz(mp, NSIDE_EXPECTED, crow["time"])
                above = alt > 0
                if classify_patchy(vals[above]):
                    date_is_patchy = True
                    break
            except Exception:
                continue
        if date_is_patchy:
            patchy_dates.append(cdate)
        else:
            non_patchy_dates.append(cdate)

    print(f"      Patchy     : {len(patchy_dates)}  {patchy_dates}")
    print(f"      Non-patchy : {len(non_patchy_dates)}")
    print(f"      Patchy dates will be processed first.")

    # 4. Build work list — patchy first, then non-patchy
    ordered_dates = patchy_dates + non_patchy_dates
    work_items = []
    for cdate in ordered_dates:
        crows = cloud_df[cloud_df["date"] == cdate]
        for sched_night_id, sched_df in nights_dict.items():
            work_items.append((
                sched_night_id,
                sched_df,
                cdate,
                crows,
                str(RESULTS_DIR),
            ))

    total_pairs = len(work_items)
    print(f"\n[4/4] Launching {total_pairs} pairs "
          f"({len(patchy_dates)} patchy x {len(nights_dict)} nights first) "
          f"with {N_WORKERS} workers ...\n")

    all_results = []
    completed   = 0

    with Pool(processes=N_WORKERS) as pool:
        for batch in pool.imap_unordered(analyze_pair, work_items, chunksize=1):
            all_results.extend(batch)
            completed += 1
            pct = 100.0 * completed / total_pairs
            n_patchy_so_far = sum(r.get("is_patchy", False) for r in all_results)
            print(f"  >> {completed}/{total_pairs} pairs ({pct:.1f}%) | "
                  f"{len(all_results)} obs, {n_patchy_so_far} patchy")

            if completed % 10 == 0 or completed == total_pairs:
                update_summary(all_results, SUMMARY_CSV)

    # Final
    print("\n" + "=" * 70)
    print("  DONE — generating plots and final summary")
    print("=" * 70)

    update_summary(all_results, SUMMARY_CSV)

    if all_results:
        create_plots(all_results)
        df        = pd.DataFrame(all_results)
        patchy_df = df[df["is_patchy"]]
        print(f"\n  Total observations   : {len(df)}")
        print(f"  Patchy observations  : {len(patchy_df)}")
        if len(df) > 0:
            print(f"  Overall mean gain    : {df['gain_mag'].mean():.4f} mag")
            print(f"  Overall median gain  : {df['gain_mag'].median():.4f} mag")
        if len(patchy_df) > 0:
            print(f"  Patchy mean gain     : {patchy_df['gain_mag'].mean():.4f} mag")
            print(f"  Patchy median gain   : {patchy_df['gain_mag'].median():.4f} mag")
        print(f"\n  Output files:")
        print(f"    results/gain_<night>_<date>.csv  (per-pair detail)")
        print(f"    {SUMMARY_CSV}")
        print(f"    scheduler_gain_analysis.png")
        print(f"    scheduler_gain_timeseries.png")
    else:
        print("  No results. Check CLOUD_CSV paths and DB_FILE.")


if __name__ == "__main__":
    main()

  SCHEDULER vs CLOUD ANALYSIS  (Rubin USDF)
  Workers  : 32
  FoV bin  : 4 deg x 4 deg
  Patchy   : SD > 0.15 mag  &  mean > 0.1 mag
  Time win : 15.0 min

[1/4] Loading cloud index from feb5_data.csv ...
      767369 maps across 198 dates: 2025-06-25 to 2026-02-05

[2/4] Loading scheduler nights from baseline_v5.1.0_10yrs.db ...
      2771 nights (#0 to #3651)

[3/4] Pre-classifying 198 cloud dates for patchiness ...
